Now try to run on a more diverse dataset of trajectories

Doesn't work very well.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# diffusion policy import
from typing import Tuple, Sequence, Dict, Union, Optional
import numpy as np
import torch
import torch.nn as nn
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler

# Painting imports
import cv2
from style.diffusion_policy_gml.dataset import PushTStateDataset
from style.diffusion_policy_gml.network import MemorizationModel, ConditionalUnet1D, compute_noise, compute_orig
from style.diffusion_policy_gml.env import PaintingEnv
import style.diffusion_policy_gml.network as network

# General
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter

In [ ]:
pred_horizon = 128
obs_horizon = 128  # pad end but don't pad start, to discourage standing still at start
obs_horizon = 1  # pad end but don't pad start, to discourage standing still at start
action_horizon = 1
obs_dim = 0  # x/y
action_dim = 2  # dx/dy

num_diffusion_iters = 100

## Load Dataset

In [ ]:
# dataset_path = "data/gml_000000.zarr"
dataset_path = "data/gml_003000.zarr"

# create dataset from file
dataset = PushTStateDataset(
    dataset_path=dataset_path,
    pred_horizon=pred_horizon,
    obs_horizon=obs_horizon,
    action_horizon=action_horizon,
    action_delta=True
)
print(dataset.indices.shape)
# print(dataset.episode_ends)
# print(dataset.indices)

# create dataloader
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=256,
    num_workers=1,
    shuffle=True,
    pin_memory=True,
    persistent_workers=True
)

# visualize data in batch
print("Num batches:          ", len(list(iter(dataloader))))
batch = next(iter(dataloader))
print("batch['obs'].shape:   ", batch['obs'].shape)
print("batch['action'].shape:", batch['action'].shape)

In [ ]:
fig, axes = plt.subplots(len(batch.keys()), 1, figsize=(10, 6))

batch = next(iter(dataloader))
for i in range(10):
    obs = np.cumsum(batch['action'][i], axis=0) + torch.tensor([10 * i, 0])
    axes[0].plot(*obs.T, '.-')
    axes[0].set_title('obs')
    # axes[1].plot(*batch['action'][i].T, '.-')
    axes[1].plot(batch['action'][i], '-')
    axes[1].set_title('action')
# for ax in axes:
#     ax.axis('equal')
axes[0].axis('equal')

In [ ]:
len(dataset.episode_ends) // 100, len(dataset.episode_ends)

In [ ]:
len(dataset.episode_ends[::(len(dataset.episode_ends) // 100)])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i in range(1, 20):
    s, e = dataset.episode_ends[i - 1], dataset.episode_ends[i]
    axes[0].plot(*dataset.normalized_train_data['obs'][s:e].T, '.-')
    axes[0].axis('equal')
for i in range(1, len(dataset.episode_ends), len(dataset.episode_ends) // 100):
    s, e = dataset.episode_ends[i - 1], dataset.episode_ends[i]
    axes[1].plot(*dataset.normalized_train_data['obs'][s:e].T, '.-')
    axes[1].axis('equal')

## Diffusion Setup

In [ ]:
# Noise scheduler
noise_scheduler = DDPMScheduler(
    num_train_timesteps=num_diffusion_iters,
    # the choise of beta schedule has big impact on performance
    # we found squared cosine works the best
    beta_schedule='squaredcos_cap_v2',
    # clip output to [-1,1] to improve stability
    clip_sample=True,
    clip_sample_range=5,
    # our network predicts noise (instead of denoised action)
    prediction_type='epsilon'
)

In [ ]:
# Network!
noise_pred_net = ConditionalUnet1D(
    input_dim=action_dim,
    global_cond_dim=0,
)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.zeros_(m.bias)
noise_pred_net.apply(init_weights);

In [ ]:
# Test with example inputs
noised_action = torch.randn((1, pred_horizon, action_dim))
obs = torch.zeros((1, obs_horizon, obs_dim))
diffusion_iter = torch.zeros((1,), dtype=torch.long)

# compute
noise = noise_pred_net(
    sample=noised_action,
    timestep=diffusion_iter,
    global_cond=obs.flatten(start_dim=1))

# check denoising
denoised_action = noised_action - noise

# device transfer
device = torch.device('cuda')
_ = noise_pred_net.to(device)

## Training

In [ ]:
num_epochs = 500 // len(list(iter(dataloader))) + 1

ema = EMAModel(
    parameters=noise_pred_net.parameters(),
    model=noise_pred_net,
    power=0.75)

optimizer = torch.optim.AdamW(
    params=noise_pred_net.parameters(),
    lr=1e-4, weight_decay=1e-6)

lr_scheduler = get_scheduler(
    name='cosine',
    optimizer=optimizer,
    num_warmup_steps=500,
    num_training_steps=len(dataloader) * num_epochs
)

writer = SummaryWriter()  # Tensorboard

if True:
    noise_pred_net.apply(init_weights)

with tqdm(range(num_epochs), desc='Epoch') as tglobal:
    all_losses = list()
    # epoch loop
    for epoch_idx in tglobal:
        epoch_loss = list()
        with tqdm(dataloader, desc='Batch') as tdataloader:
            # batch loop
            for batch_n in tdataloader:
                # Extract data
                obs_n = batch_n['obs'].to(device)
                action_n = batch_n['action'].to(device)
                B = obs_n.shape[0]
                # assert obs_n.shape[1] == obs_horizon
                # global_cond = obs_n.flatten(start_dim=1)
                global_cond = None

                # sample noise to add to actions
                noise = torch.randn(action_n.shape, device=device)
                timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (B,), device=device).long()
                noisy_actions = noise_scheduler.add_noise(action_n, noise, timesteps)

                # predict the noise residual
                noise_pred = noise_pred_net(noisy_actions, timesteps, global_cond=global_cond)

                # L2 loss
                loss = nn.functional.mse_loss(noise_pred, noise)

                # optimize
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                lr_scheduler.step()

                ema.step(noise_pred_net)

                # logging
                loss_cpu = loss.item()
                tdataloader.set_postfix(loss=loss_cpu)
                epoch_loss.append(loss_cpu)
                writer.add_scalar('Loss', loss_cpu, global_step=len(all_losses) + len(epoch_loss))

        tglobal.set_postfix(loss=np.mean(epoch_loss))
        all_losses.extend(epoch_loss)
        writer.add_scalar('Loss/Epoch', np.mean(epoch_loss), global_step=epoch_idx)
writer.close()

In [ ]:
# Weights of the EMA model is used for inference
ema_noise_pred_net = ConditionalUnet1D(
    input_dim=action_dim,
    global_cond_dim=obs_dim*obs_horizon,
)
ema_noise_pred_net.to(device)
ema.copy_to(ema_noise_pred_net.parameters())

# Plot the loss
plt.figure(figsize=(10, 3))
plt.semilogy(all_losses)
plt.title('Loss')

## Inference

In [ ]:
B = 15  # num samples
action_n_init = torch.randn((B, pred_horizon, action_dim), device=device)
# action_n_init = torch.randn((B, 80, action_dim), device=device)
history = []

action_n = network.eval(ema_noise_pred_net, noise_scheduler, action_n_init,
                        global_cond=global_cond[[0]] if global_cond is not None else None,
                        log_history=history)

action_n = action_n.detach().cpu().numpy()
action = dataset.unnormalize_action(action_n)

In [ ]:
def integrate(action):
    obs = np.cumsum(action, axis=0)
    obs = np.concatenate([np.zeros((1, 2)), obs], axis=0)  # prepend 0
    return obs
integrate_batch = lambda v: np.stack([integrate(a) for a in v])  # vmap having issues
obs = integrate_batch(action)

In [ ]:
gt = {'action': dataset.unnormalize_action(batch['action'][0].numpy()),
      'obs': dataset.unnormalize_obs(batch['obs'][0].numpy())}
result = {'action': action,
          'obs': obs + gt['obs'][0]}

In [ ]:
# Plot trajectories
r, c = (B - 1) // 5 + 1, 5
fig, axes = plt.subplots(r, c, figsize=(12, 1.5 * r))
print(result['obs'].shape)
for obs, ax in zip(result['obs'], axes.flatten()):
    ax.plot(*(obs - obs[0]).T, 'k.-')
    ax.axis('equal')

In [ ]:
# Plot state & action for all trajectories

fig, axes = plt.subplots(len(batch.keys()), B, figsize=(4 * B, 6))
for i, axes_ in enumerate(axes.T):
    for ax, (key, val) in zip(axes_, reversed(gt.items())):
        if i == 0:
            print(f'{key:6s}: {val.shape} {result[key][i].shape}')
        # ax.plot(val, 'k.-')
        # ax.plot(result[key][i], 'r.-')
        if key == 'obs':
            # ax.plot(*val.T, 'k.-')
            ax.plot(*result[key][i].T, 'k.-')
        else:
            # ax.plot(val, 'k-')
            ax.plot(result[key][i, :, 0], 'r-')
            ax.plot(result[key][i, :, 1], 'g-')
        ax.axis('equal')
        ax.set_title(key)

## Visualize Denoising

In [ ]:
def convert_to_action_obs(action_n):
    action = dataset.unnormalize_action(action_n)
    obs = integrate(action) + gt['obs'][0]
    return dict(action=action, obs=obs)

results = [[convert_to_action_obs(action_n_) for action_n_ in action_n] for action_n in history]

In [ ]:
# # Get aspect ratio

# fig, axes = plt.subplots(1, 2, figsize=(10, 3))
# axes[0].axis('equal')
# from operator import sub
# def get_aspect(ax):
#     # Total figure size
#     figW, figH = ax.get_figure().get_size_inches()
#     # Axis size on figure
#     _, _, w, h = ax.get_position().bounds
#     # Ratio of display units
#     disp_ratio = (figH * h) / (figW * w)
#     return disp_ratio
# print(get_aspect(axes[0]))

In [ ]:
# Create animation
outfolder = Path('results/gerry09_batch')
outfolder.mkdir(exist_ok=True)

def plot_frame(axes, result, gt, t):
    for res, ax in zip(result, axes.flatten()):
        ax.clear()
        obs = res['obs']
        ax.plot(*(obs - obs[0]).T, 'k.-')
        ax.axis('equal')


r, c = (B - 1) // 5 + 1, 5
fig, axes = plt.subplots(r, c, figsize=(12, 1.5 * r))
for t, result in enumerate(tqdm(results)):
    plot_frame(axes, result, gt, t)
    fig.savefig(outfolder / f'frame_{t:03d}.png')

In [ ]:
# Create video
# !/home/gchen328/miniconda3/bin/ffmpeg -y -r 25 -i results/gerry09/frame_%03d.png -c:v h264 -pix_fmt yuv420p results/gerry09.mp4 -hide_banner -loglevel error
!/home/gchen328/miniconda3/bin/ffmpeg -y -r 25 -i results/gerry09_batch/frame_%03d.png -c:v h264 -pix_fmt yuv420p results/gerry09_batch.mp4 -hide_banner -loglevel error

In [ ]:
# Display
from IPython.display import Video
Video("results/gerry09_batch.mp4", width=512, height=256)